# UnSmile 데이터셋을 활용한 악플/혐오 표현 탐지 모델 학습 (Colab용)

이 노트북은 **S14P11D105** 프로젝트의 한국어 혐오 표현 탐지를 위해 `beomi/KcELECTRA-base-v2022` 모델을 UnSmile 데이터셋으로 Fine-tuning하는 과정을 담고 있습니다.

### 실행 전 준비사항
1. 좌측 메뉴의 '파일' 탭에서 `unsmile_train_v1.0.tsv`와 `unsmile_valid_v1.0.tsv` 파일을 Colab에 업로드해주세요.
   - 또는 구글 드라이브를 마운트하여 사용할 수도 있습니다.
2. 런타임 유형이 **GPU**로 설정되어 있는지 확인하세요. (런타임 > 런타임 유형 변경 > 하드웨어 가속기: GPU)

In [ ]:
# [1] 필수 라이브러리 설치
!pip install -q transformers accelerate emoji soynlp scikit-learn pandas

In [ ]:
# [2] 라이브러리 임포트 및 시드 고정
import os
import random
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AdamW, get_linear_schedule_with_warmup
from sklearn.metrics import f1_score
from tqdm.auto import tqdm

def seed_everything(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True

SEED = 42
seed_everything(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
# [3] 설정 및 하이퍼파라미터
# 파일 경로는 Colab 파일 업로드 기준 '/content/' 입니다.
# 구글 드라이브 사용 시 경로를 수정하세요 (예: '/content/drive/MyDrive/...')

TRAIN_FILE = '/content/unsmile_train_v1.0.tsv'
VALID_FILE = '/content/unsmile_valid_v1.0.tsv'

MODEL_NAME = 'beomi/KcELECTRA-base-v2022'
BATCH_SIZE = 32
EPOCHS = 5
LEARNING_RATE = 2e-5
MAX_LEN = 128

In [ ]:
# [4] 데이터 로드
# TSV 파일이므로 sep='\t'를 사용합니다.

try:
    train_df = pd.read_csv(TRAIN_FILE, sep='\t')
    val_df = pd.read_csv(VALID_FILE, sep='\t')
    print("데이터 로드 성공!")
    print(f"Train set size: {len(train_df)}")
    print(f"Valid set size: {len(val_df)}")
except FileNotFoundError:
    print("❌ 파일을 찾을 수 없습니다. 좌측 폴더 아이콘을 눌러 파일을 업로드했는지 확인해주세요.")

# 라벨 컬럼 정의 (데이터셋의 실제 컬럼명)
LABEL_COLUMNS = ['여성/가족', '남성', '성소수자', '인종/국적', '연령', '지역', '종교', '기타 혐오', '악플/욕설', 'clean', '개인지칭']
num_labels = len(LABEL_COLUMNS)

print("\n사용할 라벨:", LABEL_COLUMNS)
train_df.head()

In [ ]:
# [5] 데이터셋 클래스 정의
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class UnSmileDataset(Dataset):
    def __init__(self, df, tokenizer, max_len):
        self.texts = df['문장'].values
        self.labels = df[LABEL_COLUMNS].values
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, item):
        text = str(self.texts[item])
        label = self.labels[item]

        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            return_token_type_ids=False,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt',
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.float)
        }

# 데이터셋 생성
train_dataset = UnSmileDataset(train_df, tokenizer, MAX_LEN)
val_dataset = UnSmileDataset(val_df, tokenizer, MAX_LEN)

# 데이터 로더
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
# [6] 모델 준비
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, 
    num_labels=num_labels, 
    problem_type="multi_label_classification"
)
model.to(device)

In [ ]:
# [7] 학습 및 검증 함수
def train_epoch(model, data_loader, loss_fn, optimizer, device, scheduler):
    model = model.train()
    losses = []
    
    for d in tqdm(data_loader, desc="Training"):
        input_ids = d["input_ids"].to(device)
        attention_mask = d["attention_mask"].to(device)
        targets = d["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        
        loss = loss_fn(outputs.logits, targets)
        losses.append(loss.item())
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()
    
    return np.mean(losses)

def eval_model(model, data_loader, loss_fn, device):
    model = model.eval()
    losses = []
    preds = []
    real_targets = []
    
    with torch.no_grad():
        for d in tqdm(data_loader, desc="Evaluating"):
            input_ids = d["input_ids"].to(device)
            attention_mask = d["attention_mask"].to(device)
            targets = d["labels"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )
            loss = loss_fn(outputs.logits, targets)
            losses.append(loss.item())
            
            preds.append(torch.sigmoid(outputs.logits).cpu().detach().numpy())
            real_targets.append(targets.cpu().detach().numpy())
            
    return np.mean(losses), np.vstack(preds), np.vstack(real_targets)

In [ ]:
# [8] 학습 실행
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, correct_bias=False)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=0,
    num_training_steps=total_steps
)
loss_fn = torch.nn.BCEWithLogitsLoss()

print("🚀 학습을 시작합니다...")

best_f1 = 0

for epoch in range(EPOCHS):
    print(f'Epoch {epoch + 1}/{EPOCHS}')
    print('-' * 10)

    train_loss = train_epoch(
        model,
        train_loader,
        loss_fn,
        optimizer,
        device,
        scheduler
    )

    val_loss, preds, val_targets = eval_model(
        model,
        val_loader,
        loss_fn,
        device
    )
    
    # 0.5 기준으로 라벨 예측
    final_preds = (preds > 0.5).astype(int)
    val_f1 = f1_score(val_targets, final_preds, average='macro')

    print(f'Train loss {train_loss:.4f} | Val loss {val_loss:.4f}')
    print(f'Val Macro F1 Score : {val_f1:.4f}')

    if val_f1 > best_f1:
        print("⭐ 성능 향상! 모델 저장합니다.")
        best_f1 = val_f1
        torch.save(model.state_dict(), 'best_model.pt')

In [ ]:
# [9] 모델 테스트 (추론 예시)
def predict_sentence(sentence):
    model.eval()
    encoding = tokenizer.encode_plus(
        sentence,
        add_special_tokens=True,
        max_length=MAX_LEN,
        return_token_type_ids=False,
        padding='max_length',
        truncation=True,
        return_attention_mask=True,
        return_tensors='pt',
    )

    input_ids = encoding['input_ids'].to(device).flatten().unsqueeze(0)
    attention_mask = encoding['attention_mask'].to(device).flatten().unsqueeze(0)

    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        probs = torch.sigmoid(outputs.logits).cpu().numpy()[0]

    print(f"문장: {sentence}")
    for label, prob in zip(LABEL_COLUMNS, probs):
        if prob > 0.5:
            print(f" - {label}: {prob*100:.2f}%")

# 저장된 모델 불러오기 (선택사항)
# model.load_state_dict(torch.load('best_model.pt'))

predict_sentence("이런 쓰레기 같은 영화는 처음 본다.")
predict_sentence("정말 유익하고 좋은 정보 감사합니다!")